# <b style='font-size:30px;font-family:Arial'>Create a File-Embedding-Based Collection using JSON Files</b>

File-embedding-based vector collections are designed for scenarios where you already have pre-computed embeddings in your JSON files. The workflow is:
1. Configure the Ingestor pipeline with `.files()` and `.upsert()`
2. Execute the pipeline with `.run()` to create collection and ingest embeddings
3. Wait for collection indexing to complete
4. Perform similarity searches

**Key Learning:** For FILE-EMBEDDING-BASED collections, use `embedding_columns` parameter in `ExtractionSchema` and the method chain `.files().upsert().run()` where embeddings are already present in the files.

## <b style='font-size:20px;font-family:Arial'>1. Import Required Libraries</b>

In [ ]:
# Import Required Libraries
import sys
import os
import json
import glob
import getpass
from datetime import datetime

# TeradataML imports
from teradataml import create_context, remove_context, DataFrame

# Teradata Vector Store V2 API - Ingestor workflow
from teradatagenai import Collection, CollectionManager, ColumnInfo
from teradatagenai import BasicIngestor, ExtractionSchema, TeradataAI
from teradatagenai import LocalConfig, S3Config, AzureBlobConfig
from teradatagenai.vector_store.Ingestor import Ingestor
from teradatagenai.common.constants import CollectionType
from teradatasqlalchemy.types import VARCHAR, CLOB

print("✅ All imports successful!")
print("✓ Using Vector Store V2 API - Ingestor stepwise pipeline")
print("✓ Collection Type: FILE-EMBEDDING-BASED")

✅ All imports successful!
✓ Using Vector Store V2 API - Ingestor stepwise pipeline
✓ Collection Type: FILE-EMBEDDING-BASED


## <b style='font-size:20px;font-family:Arial'>2. Connect to Teradata Vantage</b>

In [ ]:
# Configure connection parameters
os.environ['TD_HOST'] = getpass.getpass('Enter Teradata Host: ')
os.environ['TD_USERNAME'] = getpass.getpass('Enter Teradata Username: ')
os.environ['TD_PASSWORD'] = getpass.getpass('Enter Teradata Password: ')
os.environ['TD_BASE_URL'] = getpass.getpass('Enter Base URL: ')
os.environ['TD_AUTH_MECH'] = 'BASIC'

# Create connection context
con = create_context()
print(f"✓ Connected to Teradata Vantage at {os.environ['TD_HOST']}")

# Verify service health
try:
    health = CollectionManager.health()
    print("✓ Vector Store service is healthy")
except Exception as e:
    print(f"Service status: {e}")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatasqlalchemy\telemetry\queryband.py:557: UserWarning: [Teradata][teradataml](TDML_2002) Overwriting an existing context associated with Teradata Vantage Connection. Most of the operations on any teradataml DataFrames created before this will not work.
  return exposed_func(*args, **kwargs)


Authentication token is generated and set for the session.
✓ Connected to Teradata Vantage at 10.27.178.11
✓ Vector Store service is healthy


## <b style='font-size:20px;font-family:Arial'>3. Configure Embedding Model</b>

Specify the embedding model that will generate vector embeddings from the text content.

In [ ]:
# Configure embedding model (runtime generation)
os.environ["NVIDIA_API_KEY"] = getpass.getpass('Enter NVIDIA API Key: ')
embedding_model = TeradataAI(
    api_type="nim",
    model_name=getpass.getpass('Enter Embedding Model name: '),
    api_base=getpass.getpass('Enter Embedding Model API Base URL: ')
)

print(f"✓ Embedding Model: {embedding_model.model_name}")
print("✓ Embeddings will be generated automatically during ingestion")

✓ Embedding Model: nvidia/llama-3.2-nv-embedqa-1b-v2
✓ Embeddings will be generated automatically during ingestion


## <b style='font-size:20px;font-family:Arial'>4. Prepare JSON Input Data with Pre-computed Embeddings</b>

Load JSON files that contain pre-computed embeddings. You can load from local storage, S3, or Azure Blob Storage.

## <b style='font-size:20px;font-family:Arial'>4a. Load from Local Storage</b>

Using Local JSON files with pre-computed embeddings

In [ ]:
# Use existing combined JSON file with embeddings
notebook_dir = os.getcwd()
data_dir = os.path.join(os.path.dirname(notebook_dir), "example-data", "json_files")
json_file_path = os.path.join(data_dir, "documents_with_embeddings.json")

# Verify file exists
if not os.path.exists(json_file_path):
    raise FileNotFoundError(f"Combined JSON file not found: {json_file_path}")

print(f"✓ Loading JSON file: {os.path.basename(json_file_path)}")
print(f"✓ File location: {data_dir}")

# Verify the schema - should include 'embedding' field
with open(json_file_path, 'r') as f:
    data = json.load(f)
    # Handle both array and single object formats
    first_doc = data[0] if isinstance(data, list) else data
    print(f"\n✓ Schema verification:")
    print(f"  Fields: {list(first_doc.keys())}")
    if 'embeddings' in first_doc:
        print(f"  Embedding dimension: {len(first_doc['embeddings'])}")
    else:
        print("  ⚠ WARNING: 'embeddings' field not found!")

# Configure local file source
local_config = LocalConfig(
    files=[json_file_path],
    files_type="json"
)
print(f"\n✓ LocalConfig created for local file ingestion")

✓ Loading JSON file: documents_with_embeddings.json
✓ File location: ./sample_data/test_data_ingestor_json_embedding

✓ Schema verification:
  Fields: ['id', 'doc_title', 'text', 'category', 'author', 'publish_date', 'embeddings']
  Embedding dimension: 2048

✓ LocalConfig created for local file ingestion


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_local = f"json_file_embedding_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema (includes embedding column for pre-computed embeddings)
extraction_schema_local = ExtractionSchema(
    table_name=f"json_embedding_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000)),
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ],
    embedding_columns=ColumnInfo(name="embeddings")  # Store embeddings as CLOB
)

# Build Ingestor pipeline (FILE_EMBEDDING_BASED: use embedding_model=None with embedding_column)
ingestor_local = (
    Ingestor(
        name=collection_name_local,
        type=CollectionType.FILE_EMBEDDING_BASED,
        target_database="vsdemo01",
        description="File-Embedding-Based collection from JSON files with pre-computed embeddings",
        embedding_model=embedding_model
    )
    .files(
        files=local_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_local
    )
    .upsert()
)

print(f"✓ Collection Name: {collection_name_local}")
print("✓ Collection Type: FILE-EMBEDDING-BASED")
print(f"✓ Storage: {type(local_config).__name__}")
print("✓ Ingestor pipeline configured (using pre-computed embeddings)")

✓ Collection Name: json_file_embedding_20260217_001213
✓ Collection Type: FILE-EMBEDDING-BASED
✓ Storage: LocalConfig
✓ Ingestor pipeline configured (using pre-computed embeddings)


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [7]:
# Check if collection already exists and destroy it
try:
    existing_collection_local = Collection(name=collection_name_local)
    if existing_collection_local.exists:
        print(f"⚠ Collection '{collection_name_local}' already exists. Destroying it first...")
        existing_collection_local.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)  # Wait for cleanup
except Exception as e:
    pass

# Execute ingestion (creates collection and ingests embeddings)
print("Starting ingestion pipeline...")
result_local = ingestor_local.run()
print("✓ Ingestion completed - embeddings uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/1

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_local = existing_collection_local.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

# Access the actual results
results_list = search_results_local.similar_objects
result_count = search_results_local.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Text: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...

2. Embedding Models Comparison
   Author: David Lee
   Category: ml
   Text: Popular embedding models include sentence-transformers like all-MiniLM-L6-v2, OpenAI text-embedding-ada-002, and Cohere embed models. Each offers diff...

3. Understanding Semantic Search
   Author: John Doe
   Category: nlp
   Text: Semantic search leverages natural language processing and machine learning to understand query intent and contextual meaning. Unlike traditional keywo...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_local.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_local}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_json_file_embedding_20260217_001213_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,json_embedding_table_20260217001213,1,documents_with_embeddings.json,"0.03680419921875,0.00651931762695312,0.00774002075195312,-0.0201263427734375,-0.0181884765625,-0.01116943359375,-0.0137252807617188,-0.00208473205566406,0.04742431640625,0.03717041015625,0.00972747802734375,0.0128021240234375,-0.024566650390625,0.00732040405273438,0.021759033203125,-0.01959228515625,0.020782470703125,0.000884056091308594,-0.01837158203125,-0.00386238098144531,0.034759521484375,-0.00780487060546875,0.0210723876953125,-0.00237655639648438,-0.018341064453125,0.017608642578125,0.034393310546875,0.05328369140625,-0.00665283203125,0.0260772705078125,-0.0151443481445312,-0.04205322265625,0.0111923217773438,-0.0112991333007812,-0.0272216796875,0.01214599609375,0.0005645751953125,0.0150146484375,0.029083251953125,-0.00928497314453125,0.0223541259765625,0.0018463134765625,0.017822265625,-0.033935546875,-0.0130996704101562,-0.026123046875,0.042388916015625,0.0139236450195312,-0.0047149658203125,0.00182056427001953,0.024810791015625,-0.00921630859375,0.00518417358398438,-0.0270843505859375,-0.00279426574","0.0367938801646233,0.00651748990640044,0.00773785077035427,-0.0201207008212805,-0.0181833766400814,-0.0111663024872541,-0.0137214325368404,-0.00208414765074849,0.0474110208451748,0.0371599905192852,0.00972475111484528,0.0127985347062349,-0.0245597623288631,0.00731835188344121,0.0217529330402613,-0.0195867922157049,0.0207766443490982,0.000883808243088424,-0.0183664318174124,-0.00386129808612168,0.0347497761249542,-0.00780268246307969,0.0210664793848991,-0.00237589003518224,-0.018335921689868,0.0176037065684795,0.0343836694955826,0.0532687529921532,-0.00665096705779433,0.0260699596256018,-0.0151401022449136,-0.0420414321124554,0.0111891841515899,-0.011295965872705,-0.0272140484303236,0.0121425911784172,0.000564416928682476,0.0150104388594627,0.0290750991553068,-0.0092823700979352,0.0223478581756353,0.00184579589404166,0.0178172681480646,-0.0339260324835777,-0.0130959982052445,-0.0261157229542732,0.0423770323395729,0.013919741846621,-0.00471364380791783,0.00182005390524864,0.0248038358986378,-0.00921372510492802"
vsdemo01,json_embedding_table_20260217001213,2,documents_with_embeddings.json,"-0.00275611877441406,-0.037445068359375,0.0130081176757812,-0.00722122192382812,0.0272674560546875,-0.00593948364257812,0.000760555267333984,-0.0209808349609375,0.0095367431640625,-0.00284194946289062,-0.0167999267578125,0.00505828857421875,0.0152359008789062,0.00398635864257812,-0.000733375549316406,-0.0216827392578125,-0.000119268894195557,-0.0207061767578125,-0.0206756591796875,0.01141357421875,0.017730712890625,-0.010589599609375,0.00245094299316406,-0.0241546630859375,-0.027313232421875,0.00557327270507812,-0.00809478759765625,0.042449951171875,-0.029937744140625,-0.00319290161132812,0.0131378173828125,-0.00317001342773438,0.042999267578125,-0.00521469116210938,0.0187835693359375,-0.020538330078125,0.0116958618164062,0.0271759033203125,-0.029052734375,-0.0067291259765625,0.03924560546875,0.00149059295654297,0.0283660888671875,-0.002655029296875,-0.023834228515625,-0.047637939453125,0.048370361328125,0.00342559814453125,-0.00909423828125,-0.033294677734375,-0.009490966796875,-0.000916481018066406,-0.00670","-0.00275701587088406,-0.0374572575092316,0.0130123514682055,-0.00722357258200645,0.0272763315588236,-0.00594141706824303,0.000760802824515849,-0.0209876634180546,0.00953984726220369,-0.00284287449903786,-0.0168053954839706,0.00505993515253067,0.0152408601716161,0.00398765597492456,-0.000733614258933812,-0.0216897968202829,-0.000119307718705386,-0.0207129158079624,-0.0206823889166117,0.0114172892645001,0.0177364833652973,-0.0105930464342237,0.00245174067094922,-0.0241625253111124,-0.0273221228271723,0.00557508692145348,-0.00809742230921984,0.042463768273592,-0.0299474895000458,-0.0031939409673214,0.0131420940160751,-0.00317104533314705,0.0430132634937763,-0.00521638849750161,0

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_local.destroy()
print(f"✓ Collection '{collection_name_local}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4b. Load from S3 Storage</b>

Instead of local files, you can ingest JSON files with pre-computed embeddings directly from Amazon S3.

In [ ]:
from teradatagenai.vector_store import S3Config

# S3 Configuration
s3_config = S3Config(
    bucket=getpass.getpass('Enter S3 Bucket: '),
    key=getpass.getpass('Enter S3 Key: '),
    aws_access_key_id=getpass.getpass('Enter AWS Access Key ID: '),
    aws_secret_access_key=getpass.getpass('Enter AWS Secret Access Key: '),
    region_name=getpass.getpass('Enter AWS Region: '),
    files_type="json",
    aws_session_token=getpass.getpass('Enter AWS Session Token: ')
)

print("✓ S3 storage configuration enabled")
print(f"  Bucket: {s3_config.bucket}")
print(f"  Key: {s3_config.key}")
print(f"  Region: {s3_config.region_name}")

✓ S3 storage configuration enabled
  Bucket: genaitestaanchal
  Key: files/documents_with_embeddings.json
  Region: us-west-2


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_s3 = f"json_file_embedding_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_s3 = ExtractionSchema(
    table_name=f"json_embedding_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ],
    embedding_columns=ColumnInfo(name="embeddings")  # Store embeddings as VARCHAR (or CLOB if larger)
)

# Build Ingestor pipeline
ingestor_s3 = (
    Ingestor(
        name=collection_name_s3,
        type=CollectionType.FILE_EMBEDDING_BASED,
        target_database="vsdemo01",
        description="File-Embedding-Based collection from JSON files with pre-computed embeddings",
        embedding_model=embedding_model
    )
    .files(
        files=s3_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_s3
    )
    .upsert()
)

print(f"✓ Collection Name: {collection_name_s3}")
print("✓ Collection Type: FILE-EMBEDDING-BASED")
print(f"✓ Storage: {type(s3_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: json_file_embedding_20260217_001810
✓ Collection Type: FILE-EMBEDDING-BASED
✓ Storage: S3Config
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [15]:
# Check if collection already exists and destroy it
try:
    existing_collection_s3 = Collection(name=collection_name_s3)
    if existing_collection_s3.exists:
        print(f"⚠ Collection '{collection_name_s3}' already exists. Destroying it first...")
        existing_collection_s3.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_s3 = ingestor_s3.run()
print("✓ Ingestion completed - embeddings uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'json_file_embedding_20260217_001810' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_s3 = existing_collection_s3.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_s3.similar_objects
result_count = search_results_s3.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Text: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...

2. Embedding Models Comparison
   Author: David Lee
   Category: ml
   Text: Popular embedding models include sentence-transformers like all-MiniLM-L6-v2, OpenAI text-embedding-ada-002, and Cohere embed models. Each offers diff...

3. Understanding Semantic Search
   Author: John Doe
   Category: nlp
   Text: Semantic search leverages natural language processing and machine learning to understand query intent and contextual meaning. Unlike traditional keywo...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_s3.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_s3}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_json_file_embedding_20260217_001810_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,json_embedding_table_20260217001810,1,documents_with_embeddings.json,"0.03680419921875,0.00651931762695312,0.00774002075195312,-0.0201263427734375,-0.0181884765625,-0.01116943359375,-0.0137252807617188,-0.00208473205566406,0.04742431640625,0.03717041015625,0.00972747802734375,0.0128021240234375,-0.024566650390625,0.00732040405273438,0.021759033203125,-0.01959228515625,0.020782470703125,0.000884056091308594,-0.01837158203125,-0.00386238098144531,0.034759521484375,-0.00780487060546875,0.0210723876953125,-0.00237655639648438,-0.018341064453125,0.017608642578125,0.034393310546875,0.05328369140625,-0.00665283203125,0.0260772705078125,-0.0151443481445312,-0.04205322265625,0.0111923217773438,-0.0112991333007812,-0.0272216796875,0.01214599609375,0.0005645751953125,0.0150146484375,0.029083251953125,-0.00928497314453125,0.0223541259765625,0.0018463134765625,0.017822265625,-0.033935546875,-0.0130996704101562,-0.026123046875,0.042388916015625,0.0139236450195312,-0.0047149658203125,0.00182056427001953,0.024810791015625,-0.00921630859375,0.00518417358398438,-0.0270843505859375,-0.00279426574","0.0367938801646233,0.00651748990640044,0.00773785077035427,-0.0201207008212805,-0.0181833766400814,-0.0111663024872541,-0.0137214325368404,-0.00208414765074849,0.0474110208451748,0.0371599905192852,0.00972475111484528,0.0127985347062349,-0.0245597623288631,0.00731835188344121,0.0217529330402613,-0.0195867922157049,0.0207766443490982,0.000883808243088424,-0.0183664318174124,-0.00386129808612168,0.0347497761249542,-0.00780268246307969,0.0210664793848991,-0.00237589003518224,-0.018335921689868,0.0176037065684795,0.0343836694955826,0.0532687529921532,-0.00665096705779433,0.0260699596256018,-0.0151401022449136,-0.0420414321124554,0.0111891841515899,-0.011295965872705,-0.0272140484303236,0.0121425911784172,0.000564416928682476,0.0150104388594627,0.0290750991553068,-0.0092823700979352,0.0223478581756353,0.00184579589404166,0.0178172681480646,-0.0339260324835777,-0.0130959982052445,-0.0261157229542732,0.0423770323395729,0.013919741846621,-0.00471364380791783,0.00182005390524864,0.0248038358986378,-0.00921372510492802"
vsdemo01,json_embedding_table_20260217001810,6,documents_with_embeddings.json,"-0.05364990234375,-0.0137481689453125,-0.00927734375,0.0125045776367188,0.000814914703369141,-0.000749111175537109,0.0262451171875,-0.007965087890625,0.050140380859375,0.0104293823242188,-0.017242431640625,0.00732803344726562,0.0095367431640625,0.0115890502929688,-0.000130057334899902,-0.0172576904296875,-0.0051727294921875,-0.010101318359375,-0.0289306640625,0.0038604736328125,0.0054779052734375,-0.0516357421875,-0.020172119140625,-0.00341987609863281,0.00121021270751953,0.0238189697265625,-0.0069122314453125,0.00641250610351562,0.0243988037109375,0.0096588134765625,0.0091705322265625,-0.0078887939453125,-0.0347900390625,0.013946533203125,-0.0200042724609375,-0.0130844116210938,-0.00457382202148438,-0.0232391357421875,0.0189208984375,-0.037200927734375,0.03594970703125,0.0130462646484375,0.015777587890625,-0.00220108032226562,-0.00634002685546875,-0.0290985107421875,0.0195465087890625,0.00583648681640625,0.0115280151367188,0.00134944915771484,0.03448486328125,-0.016998291015625,0.00596237182617188,-0.0062179","-0.0536545813083649,-0.0137493684887886,-0.00927815306931734,0.0125056682154536,0.000814985774923116,-0.000749176542740315,0.0262474063783884,-0.00796578265726566,0.0501447543501854,0.0104302922263741,-0.0172439366579056,0.0073286728002131,0.00953757483512163,0.0115900617092848,-0.00013006868539378,-0.0172591954469681,-0.0051731807179749,-0.0101021993905306,-0.0289331879466772,0.00386081030592322,0.00547838304191828,-0.0516402460634708,-0.020173879340291,-0.00342017435468733,0.00121031829621643,0.023821048438549,-0.00691283447667956,0.00641306536272168,0.0244009327143431,0.00965965632349253,0.00917133223265409,-0.00788948219269514,-0.0347930751740932,0.0139477495104074,-0.020

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_s3.destroy()
print(f"✓ Collection '{collection_name_s3}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4c. Load from Azure Blob Storage</b>

Instead of local files or S3, you can ingest JSON files with pre-computed embeddings from Azure Blob Storage.

In [ ]:
from teradatagenai.vector_store import AzureBlobConfig

# Azure Blob Configuration
azure_config = AzureBlobConfig(
    container=getpass.getpass('Enter Azure Container: '),
    blob_name=getpass.getpass('Enter Blob Name: '),
    account_name=getpass.getpass('Enter Azure Account Name: '),
    account_key=getpass.getpass('Enter Azure Account Key: '),
    files_type="json"
)

print("✓ Azure Blob storage configuration provided")
print(f"  Container: {azure_config.container}")
print(f"  Blob: {azure_config.blob_name}")
print(f"  Account: {azure_config.account_name}")

✓ Azure Blob storage configuration provided
  Container: genaitestcontainer
  Blob: documents_with_embeddings.json
  Account: genaitestaanchal


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_blob = f"json_file_embedding_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_blob = ExtractionSchema(
    table_name=f"json_embedding_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ],
    embedding_columns=ColumnInfo(name="embeddings")  # Store embeddings as VARCHAR (or CLOB if larger)
)

# Build Ingestor pipeline
ingestor_blob = (
    Ingestor(
        name=collection_name_blob,
        type=CollectionType.FILE_EMBEDDING_BASED,
        target_database="vsdemo01",
        description="File-Embedding-Based collection from JSON files with pre-computed embeddings",
        embedding_model=embedding_model
    )
    .files(
        files=azure_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_blob
    )
    .upsert()
)

print(f"✓ Collection Name: {collection_name_blob}")
print("✓ Collection Type: FILE-EMBEDDING-BASED")
print(f"✓ Storage: {type(azure_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: json_file_embedding_20260217_002125
✓ Collection Type: FILE-EMBEDDING-BASED
✓ Storage: AzureBlobConfig
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [20]:
# Check if collection already exists and destroy it
try:
    existing_collection_blob = Collection(name=collection_name_blob)
    if existing_collection_blob.exists:
        print(f"⚠ Collection '{collection_name_blob}' already exists. Destroying it first...")
        existing_collection_blob.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_blob = ingestor_blob.run()
print("✓ Ingestion completed - embeddings uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'json_file_embedding_20260217_002125' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_blob = existing_collection_blob.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_blob.similar_objects
result_count = search_results_blob.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Text: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...

2. Embedding Models Comparison
   Author: David Lee
   Category: ml
   Text: Popular embedding models include sentence-transformers like all-MiniLM-L6-v2, OpenAI text-embedding-ada-002, and Cohere embed models. Each offers diff...

3. Understanding Semantic Search
   Author: John Doe
   Category: nlp
   Text: Semantic search leverages natural language processing and machine learning to understand query intent and contextual meaning. Unlike traditional keywo...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_blob.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_blob}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_json_file_embedding_20260217_002125_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,json_embedding_table_20260217002125,1,documents_with_embeddings.json,"0.03680419921875,0.00651931762695312,0.00774002075195312,-0.0201263427734375,-0.0181884765625,-0.01116943359375,-0.0137252807617188,-0.00208473205566406,0.04742431640625,0.03717041015625,0.00972747802734375,0.0128021240234375,-0.024566650390625,0.00732040405273438,0.021759033203125,-0.01959228515625,0.020782470703125,0.000884056091308594,-0.01837158203125,-0.00386238098144531,0.034759521484375,-0.00780487060546875,0.0210723876953125,-0.00237655639648438,-0.018341064453125,0.017608642578125,0.034393310546875,0.05328369140625,-0.00665283203125,0.0260772705078125,-0.0151443481445312,-0.04205322265625,0.0111923217773438,-0.0112991333007812,-0.0272216796875,0.01214599609375,0.0005645751953125,0.0150146484375,0.029083251953125,-0.00928497314453125,0.0223541259765625,0.0018463134765625,0.017822265625,-0.033935546875,-0.0130996704101562,-0.026123046875,0.042388916015625,0.0139236450195312,-0.0047149658203125,0.00182056427001953,0.024810791015625,-0.00921630859375,0.00518417358398438,-0.0270843505859375,-0.00279426574","0.0367938801646233,0.00651748990640044,0.00773785077035427,-0.0201207008212805,-0.0181833766400814,-0.0111663024872541,-0.0137214325368404,-0.00208414765074849,0.0474110208451748,0.0371599905192852,0.00972475111484528,0.0127985347062349,-0.0245597623288631,0.00731835188344121,0.0217529330402613,-0.0195867922157049,0.0207766443490982,0.000883808243088424,-0.0183664318174124,-0.00386129808612168,0.0347497761249542,-0.00780268246307969,0.0210664793848991,-0.00237589003518224,-0.018335921689868,0.0176037065684795,0.0343836694955826,0.0532687529921532,-0.00665096705779433,0.0260699596256018,-0.0151401022449136,-0.0420414321124554,0.0111891841515899,-0.011295965872705,-0.0272140484303236,0.0121425911784172,0.000564416928682476,0.0150104388594627,0.0290750991553068,-0.0092823700979352,0.0223478581756353,0.00184579589404166,0.0178172681480646,-0.0339260324835777,-0.0130959982052445,-0.0261157229542732,0.0423770323395729,0.013919741846621,-0.00471364380791783,0.00182005390524864,0.0248038358986378,-0.00921372510492802"
vsdemo01,json_embedding_table_20260217002125,6,documents_with_embeddings.json,"-0.05364990234375,-0.0137481689453125,-0.00927734375,0.0125045776367188,0.000814914703369141,-0.000749111175537109,0.0262451171875,-0.007965087890625,0.050140380859375,0.0104293823242188,-0.017242431640625,0.00732803344726562,0.0095367431640625,0.0115890502929688,-0.000130057334899902,-0.0172576904296875,-0.0051727294921875,-0.010101318359375,-0.0289306640625,0.0038604736328125,0.0054779052734375,-0.0516357421875,-0.020172119140625,-0.00341987609863281,0.00121021270751953,0.0238189697265625,-0.0069122314453125,0.00641250610351562,0.0243988037109375,0.0096588134765625,0.0091705322265625,-0.0078887939453125,-0.0347900390625,0.013946533203125,-0.0200042724609375,-0.0130844116210938,-0.00457382202148438,-0.0232391357421875,0.0189208984375,-0.037200927734375,0.03594970703125,0.0130462646484375,0.015777587890625,-0.00220108032226562,-0.00634002685546875,-0.0290985107421875,0.0195465087890625,0.00583648681640625,0.0115280151367188,0.00134944915771484,0.03448486328125,-0.016998291015625,0.00596237182617188,-0.0062179","-0.0536545813083649,-0.0137493684887886,-0.00927815306931734,0.0125056682154536,0.000814985774923116,-0.000749176542740315,0.0262474063783884,-0.00796578265726566,0.0501447543501854,0.0104302922263741,-0.0172439366579056,0.0073286728002131,0.00953757483512163,0.0115900617092848,-0.00013006868539378,-0.0172591954469681,-0.0051731807179749,-0.0101021993905306,-0.0289331879466772,0.00386081030592322,0.00547838304191828,-0.0516402460634708,-0.020173879340291,-0.00342017435468733,0.00121031829621643,0.023821048438549,-0.00691283447667956,0.00641306536272168,0.0244009327143431,0.00965965632349253,0.00917133223265409,-0.00788948219269514,-0.0347930751740932,0.0139477495104074,-0.020

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_blob.destroy()
print(f"✓ Collection '{collection_name_blob}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4d. Load from Google Cloud Storage</b>

Instead of local files, S3, or Azure, you can ingest JSON files with pre-computed embeddings from Google Cloud Platform (GCP) Storage.

In [ ]:
from teradatagenai import GCPConfig

# GCP Configuration
gcp_config = GCPConfig(
    files_type="json",
    bucket=getpass.getpass('Enter GCP Bucket: '),
    blob_name=getpass.getpass('Enter GCP Blob Name: '),
    project_id=getpass.getpass('Enter GCP Project ID: '),
    secret=json.loads(getpass.getpass('Enter GCP Service Account JSON (paste entire JSON): '))
)

print("✓ GCP storage configuration enabled")
print(f"  Bucket: {gcp_config.bucket}")
print(f"  Blob: {gcp_config.blob_name}")
print(f"  Project: {gcp_config.project_id}")

✓ GCP storage configuration enabled
  Bucket: ak250090-filestore
  Blob: harsha_files/documents_with_embeddings.json
  Project: icgcp-vector-store


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_gcp = f"json_file_embedding_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema (includes embedding column for pre-computed embeddings)
extraction_schema_gcp = ExtractionSchema(
    table_name=f"json_embedding_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ],
    embedding_columns=ColumnInfo(name="embeddings")  # Store embeddings as VARCHAR (or CLOB if larger
)

# Build Ingestor pipeline (note: uses .embed() instead of .upsert())
ingestor_gcp = (
    Ingestor(
        name=collection_name_gcp,
        type=CollectionType.FILE_EMBEDDING_BASED,
        target_database="vsdemo01",
        description="File-Embedding-Based collection from JSON files with pre-computed embeddings",
        embedding_model=embedding_model
    )
    .files(
        files=gcp_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_gcp
    )
    .upsert()
)

print(f"✓ Collection Name: {collection_name_gcp}")
print("✓ Collection Type: FILE-EMBEDDING-BASED")
print(f"✓ Storage: {type(gcp_config).__name__}")
print("✓ Ingestor pipeline configured (using pre-computed embeddings)")

✓ Collection Name: json_file_embedding_20260217_002532
✓ Collection Type: FILE-EMBEDDING-BASED
✓ Storage: GCPConfig
✓ Ingestor pipeline configured (using pre-computed embeddings)


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [25]:
# Check if collection already exists and destroy it
try:
    existing_collection_gcp = Collection(name=collection_name_gcp)
    if existing_collection_gcp.exists:
        print(f"⚠ Collection '{collection_name_gcp}' already exists. Destroying it first...")
        existing_collection_gcp.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)  # Wait for cleanup
except Exception as e:
    pass

# Execute ingestion (creates collection and ingests embeddings)
print("Starting ingestion pipeline...")
result_gcp = ingestor_gcp.run()
print("✓ Ingestion completed - embeddings uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'json_file_embedding_20260217_002532' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_gcp = existing_collection_gcp.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

# Access the actual results
results_list = search_results_gcp.similar_objects
result_count = search_results_gcp.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Jane Smith
   Category: database
   Text: Vector databases are specialized systems designed to store and query high-dimensional vector embeddings efficiently. They enable similarity search for...

2. Embedding Models Comparison
   Author: David Lee
   Category: ml
   Text: Popular embedding models include sentence-transformers like all-MiniLM-L6-v2, OpenAI text-embedding-ada-002, and Cohere embed models. Each offers diff...

3. Understanding Semantic Search
   Author: John Doe
   Category: nlp
   Text: Semantic search leverages natural language processing and machine learning to understand query intent and contextual meaning. Unlike traditional keywo...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_gcp.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_gcp}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_json_file_embedding_20260217_002532_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,json_embedding_table_20260217002532,1,documents_with_embeddings.json,"0.03680419921875,0.00651931762695312,0.00774002075195312,-0.0201263427734375,-0.0181884765625,-0.01116943359375,-0.0137252807617188,-0.00208473205566406,0.04742431640625,0.03717041015625,0.00972747802734375,0.0128021240234375,-0.024566650390625,0.00732040405273438,0.021759033203125,-0.01959228515625,0.020782470703125,0.000884056091308594,-0.01837158203125,-0.00386238098144531,0.034759521484375,-0.00780487060546875,0.0210723876953125,-0.00237655639648438,-0.018341064453125,0.017608642578125,0.034393310546875,0.05328369140625,-0.00665283203125,0.0260772705078125,-0.0151443481445312,-0.04205322265625,0.0111923217773438,-0.0112991333007812,-0.0272216796875,0.01214599609375,0.0005645751953125,0.0150146484375,0.029083251953125,-0.00928497314453125,0.0223541259765625,0.0018463134765625,0.017822265625,-0.033935546875,-0.0130996704101562,-0.026123046875,0.042388916015625,0.0139236450195312,-0.0047149658203125,0.00182056427001953,0.024810791015625,-0.00921630859375,0.00518417358398438,-0.0270843505859375,-0.00279426574","0.0367938801646233,0.00651748990640044,0.00773785077035427,-0.0201207008212805,-0.0181833766400814,-0.0111663024872541,-0.0137214325368404,-0.00208414765074849,0.0474110208451748,0.0371599905192852,0.00972475111484528,0.0127985347062349,-0.0245597623288631,0.00731835188344121,0.0217529330402613,-0.0195867922157049,0.0207766443490982,0.000883808243088424,-0.0183664318174124,-0.00386129808612168,0.0347497761249542,-0.00780268246307969,0.0210664793848991,-0.00237589003518224,-0.018335921689868,0.0176037065684795,0.0343836694955826,0.0532687529921532,-0.00665096705779433,0.0260699596256018,-0.0151401022449136,-0.0420414321124554,0.0111891841515899,-0.011295965872705,-0.0272140484303236,0.0121425911784172,0.000564416928682476,0.0150104388594627,0.0290750991553068,-0.0092823700979352,0.0223478581756353,0.00184579589404166,0.0178172681480646,-0.0339260324835777,-0.0130959982052445,-0.0261157229542732,0.0423770323395729,0.013919741846621,-0.00471364380791783,0.00182005390524864,0.0248038358986378,-0.00921372510492802"
vsdemo01,json_embedding_table_20260217002532,2,documents_with_embeddings.json,"-0.00275611877441406,-0.037445068359375,0.0130081176757812,-0.00722122192382812,0.0272674560546875,-0.00593948364257812,0.000760555267333984,-0.0209808349609375,0.0095367431640625,-0.00284194946289062,-0.0167999267578125,0.00505828857421875,0.0152359008789062,0.00398635864257812,-0.000733375549316406,-0.0216827392578125,-0.000119268894195557,-0.0207061767578125,-0.0206756591796875,0.01141357421875,0.017730712890625,-0.010589599609375,0.00245094299316406,-0.0241546630859375,-0.027313232421875,0.00557327270507812,-0.00809478759765625,0.042449951171875,-0.029937744140625,-0.00319290161132812,0.0131378173828125,-0.00317001342773438,0.042999267578125,-0.00521469116210938,0.0187835693359375,-0.020538330078125,0.0116958618164062,0.0271759033203125,-0.029052734375,-0.0067291259765625,0.03924560546875,0.00149059295654297,0.0283660888671875,-0.002655029296875,-0.023834228515625,-0.047637939453125,0.048370361328125,0.00342559814453125,-0.00909423828125,-0.033294677734375,-0.009490966796875,-0.000916481018066406,-0.00670","-0.00275701587088406,-0.0374572575092316,0.0130123514682055,-0.00722357258200645,0.0272763315588236,-0.00594141706824303,0.000760802824515849,-0.0209876634180546,0.00953984726220369,-0.00284287449903786,-0.0168053954839706,0.00505993515253067,0.0152408601716161,0.00398765597492456,-0.000733614258933812,-0.0216897968202829,-0.000119307718705386,-0.0207129158079624,-0.0206823889166117,0.0114172892645001,0.0177364833652973,-0.0105930464342237,0.00245174067094922,-0.0241625253111124,-0.0273221228271723,0.00557508692145348,-0.00809742230921984,0.042463768273592,-0.0299474895000458,-0.0031939409673214,0.0131420940160751,-0.00317104533314705,0.0430132634937763,-0.00521638849750161,0

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_gcp.destroy()
print(f"✓ Collection '{collection_name_gcp}' destroyed")

---

## Summary

This notebook demonstrated how to:
- Create a FILE-EMBEDDING-BASED vector collection from JSON files with pre-computed embeddings
- Configure the Ingestor pipeline with `embedding_columns` parameter in `ExtractionSchema`
- Test with four different storage types: Local, S3, Azure Blob, and GCP
- Implement collection readiness polling to handle async indexing
- Perform semantic similarity search on the collection

**Key Points:**
- JSON files must contain pre-computed embeddings in the `embeddings` field (plural)
- Use `embedding_columns=ColumnInfo(name="embeddings")` in `ExtractionSchema` with `.upsert()` method
- Collections require indexing time after ingestion before similarity search is available

- Metadata columns are preserved for filtering and retrieval- Each storage type (Local/S3/Azure/GCP) has its own complete isolated workflow